# Phân loại email spam - KNN 
Theo bài giảng trang 45–48: Euclidean, chọn K láng giềng, bỏ phiếu đa số.
Phần KNN tự viết bằng NumPy; không gọi KNeighborsClassifier.

## 1. Mã giả

Quy ước: RỖNG nghĩa là None/chưa có trạng thái, khác bảng rỗng {}.
Chỉ số bắt đầu từ 0; V là số đặc trưng TF-IDF. FIT không trả nhãn dự đoán.
Mã giả trình bày vòng lặp từng phần tử; code NumPy gộp một số vòng lặp thành phép tính mảng.

```text
ĐẦU VÀO: X_train (N × V), y_train (N nhãn), X_test (M × V), K
ĐẦU RA: y_pred (M nhãn)

PHƯƠNG THỨC INIT(K):
    k ← K
    ready ← SAI
    X_lưu ← RỖNG; y_lưu ← RỖNG

PHƯƠNG THỨC BUILD_MODEL():
    NẾU k không phải số nguyên HOẶC k < 1: BÁO LỖI, DỪNG
    X_lưu ← RỖNG; y_lưu ← RỖNG
    ready ← ĐÚNG

PHƯƠNG THỨC FIT(X_train, y_train):
    NẾU ready = SAI: BÁO LỖI, DỪNG
    N ← số dòng X_train
    NẾU k > N HOẶC số nhãn y_train ≠ N: BÁO LỖI, DỪNG
    X_lưu ← mảng các vector trong X_train
    y_lưu ← mảng các nhãn trong y_train

HÀM PREDICT(X_test):
    NẾU X_lưu = RỖNG: BÁO LỖI, DỪNG
    y_pred ← danh sách rỗng
    VỚI MỖI vector x TRONG X_test:
        distances ← danh sách rỗng
        VỚI i ← 0 ĐẾN số dòng X_lưu − 1:
            sum_squared ← 0
            VỚI j ← 0 ĐẾN số cột X_lưu − 1:
                sum_squared ← sum_squared + (x[j] − X_lưu[i,j])²
            THÊM (sqrt(sum_squared), i) VÀO distances
        SẮP XẾP distances tăng dần theo khoảng cách, rồi theo i
        neighbors ← k phần tử đầu của distances
        votes ← bảng có giá trị mặc định 0
        VỚI MỖI (distance, i) TRONG neighbors:
            label ← y_lưu[i]
            votes[label] ← votes[label] + 1
        labels ← các nhãn trong votes, sắp xếp tăng dần
        best ← labels[0]
        VỚI MỖI label TRONG labels:
            NẾU votes[label] > votes[best]: best ← label
        THÊM best VÀO y_pred
    TRẢ VỀ y_pred
```


In [ ]:
from pathlib import Path

# Tìm thư mục dự án từ vị trí đang mở notebook.
ROOT = Path.cwd().resolve()
while not (ROOT / "Datasets/data_raw/spam_ham_dataset.csv").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Hãy mở notebook bên trong thư mục Present_and_Lab1_ML_G5.")
    ROOT = ROOT.parent

print("Thư mục dự án:", ROOT)


## 2. Đọc và làm sạch dữ liệu
Hàm `clean_text` được viết trực tiếp trong ô code bên dưới: chuyển chữ thường, bỏ HTML/dấu câu, gộp khoảng trắng. Phân loại văn bản email tiếng Anh, không phân tích file đính kèm.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# Hàm làm sạch viết ngay tại đây, không import từ file khác.
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]*>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())

# 1. Đọc dữ liệu: chỉ lấy văn bản và nhãn.
data = pd.read_csv(ROOT / "Datasets/data_raw/spam_ham_dataset.csv")
data = data[["text", "label"]].dropna().copy()
print("Trước xử lý:", len(data))

# 2. Làm sạch và loại nội dung trùng trước khi chia tập.
data["clean_text"] = data["text"].apply(clean_text)
data = data[data["clean_text"] != ""]
if data.groupby("clean_text")["label"].nunique().max() > 1:
    raise ValueError("Có nội dung trùng nhưng khác nhãn.")
data = data.drop_duplicates(subset="clean_text")
print("Sau xử lý:", len(data))
print(data["label"].value_counts())
data[["label", "clean_text"]].head()


## 3. Chia train/test
80% để học, 20% để kiểm tra. `random_state=42` giữ lần chia cố định. `stratify` giữ tỷ lệ nhãn gần nhau.

In [ ]:
# 3. Chia văn bản trước khi học TF-IDF.
text_train, text_test, y_train, y_test = train_test_split(
    data["clean_text"], data["label"],
    test_size=0.2, random_state=42, stratify=data["label"]
)
print("Train:", len(text_train), "Test:", len(text_test))


## 4. Chuyển văn bản thành số
TF-IDF biểu diễn mỗi email bằng trọng số từ. `stop_words` bỏ từ thông dụng tiếng Anh; `max_features=1000` giới hạn số cột. Dùng bước này trước khi train Naive Bayes. Không fit trên test.

In [ ]:
# 4. Học từ vựng và trọng số từ train, chỉ biến đổi test.
vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
X_train = vectorizer.fit_transform(text_train)
X_test = vectorizer.transform(text_test)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


## 5. Xây dựng class KNN
`__init__` lưu K; `build_model` chuẩn bị trạng thái; `fit` lưu mẫu train;
`predict` tự tính khoảng cách, chọn láng giềng và đếm phiếu.
K=5 cố định, mỗi láng giềng một phiếu, không tuning trên test.
Khoảng cách: `d(x,z) = sqrt(tổng_j (x[j] - z[j])^2)`.
Trong code có bước đổi TF-IDF sang mảng để phép tính dễ đọc; bản này dùng cho lab quy mô nhỏ.


In [ ]:
import numpy as np

class KNNSpamClassifier:
    def __init__(self, k=5):
        self.k = k
        self.ready = False
        self.X_train = None
        self.y_train = None

    def build_model(self):
        if not isinstance(self.k, int) or self.k < 1:
            raise ValueError("K phải là số nguyên dương.")
        self.X_train = None
        self.y_train = None
        self.ready = True

    def fit(self, X_train, y_train):
        if not self.ready:
            raise ValueError("Hãy gọi build_model() trước fit().")
        if self.k > X_train.shape[0]:
            raise ValueError("K không được vượt quá số mẫu train.")
        if X_train.shape[0] != len(y_train):
            raise ValueError("Số mẫu và số nhãn phải bằng nhau.")
        self.X_train = X_train.toarray()
        self.y_train = np.array(y_train)

    def predict(self, X_test):
        if self.X_train is None:
            raise ValueError("Hãy gọi fit() trước predict().")
        predictions = []
        for row in range(X_test.shape[0]):
            email = X_test[row].toarray().ravel()
            differences = self.X_train - email
            distances = np.sqrt(np.sum(differences ** 2, axis=1))
            # Khoảng cách bằng nhau: giữ thứ tự mẫu trong train.
            nearest_indices = np.argsort(distances, kind="stable")[:self.k]
            nearest_labels = self.y_train[nearest_indices]
            labels, votes = np.unique(nearest_labels, return_counts=True)
            # Nếu hòa phiếu, chọn lớp đầu trong thứ tự nhãn (ham trước spam).
            best_label = labels[np.argmax(votes)]
            predictions.append(best_label)
        return np.array(predictions)


## 6. Bốn bước theo ghi chép của cô
Fit của KNN chỉ lưu các ví dụ; phép tính khoảng cách được thực hiện trong predict.

In [ ]:
model = KNNSpamClassifier(k=5)
model.build_model()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


## 7. Đánh giá
Spam là lớp dương; confusion matrix có hàng thật/cột dự đoán, thứ tự ham/spam.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix [ham, spam]:")
print(confusion_matrix(y_test, y_pred, labels=["ham", "spam"]))


## 8. Dự đoán email mới
Dùng cùng hàm clean_text và vectorizer đã fit trên train. Ví dụ demo không dùng tính điểm test.

In [ ]:
emails = ["Subject: Team meeting tomorrow. Please review the report.",
          "Subject: Free prize. Claim your money now."]
clean_emails = [clean_text(email) for email in emails]
X_new = vectorizer.transform(clean_emails)
for email, label in zip(emails, model.predict(X_new)):
    print(label, "->", email)


## 9. Lưu kết quả
Các file knn_* riêng trong outputs/, không ghi đè kết quả Naive Bayes.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

output = ROOT / "outputs"
output.mkdir(exist_ok=True)
metrics = pd.DataFrame([{
    "model": "KNN (tu viet)",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, pos_label="spam", zero_division=0),
    "recall": recall_score(y_test, y_pred, pos_label="spam", zero_division=0),
    "f1": f1_score(y_test, y_pred, pos_label="spam", zero_division=0)
}])
metrics.to_csv(output / "knn_metrics.csv", index=False)
pd.DataFrame({"source_row": y_test.index, "label": y_test.values,
              "prediction": y_pred}).to_csv(output / "knn_predictions.csv", index=False)
pd.DataFrame(confusion_matrix(y_test, y_pred, labels=["ham", "spam"]),
             index=["ham", "spam"], columns=["ham", "spam"]).to_csv(
    output / "knn_confusion_matrix.csv")
print("Đã lưu kết quả KNN vào:", output)
